<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/06_프롬프트_엔지니어링.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 6 서론](#chapter6)
* [Chapter 6-1 텍스트 생성 모델 사용하기](#chapter6-1)
    * [Chapter 6-1-1 텍스트 생성 모델 선택하기](#chapter6-1-1)
    * [Chapter 6-1-2 텍스트 생성 모델 로드하기](#chapter6-1-2)
    * [Chapter 6-1-3 모델 출력 제어하기](#chapter6-1-3)
 * [Chapter 6-2 프롬프트 엔지니어링 소개](#chapter6-2)   
     * [Chapter 6-2-1 프롬프트의 기본 구성요소](#chapter6-2-1)   
     * [Chapter 6-2-2 지시 기반 프롬프트](#chapter6-2-2)  
 * [Chapter 6-3 고급 프롬프트 엔지니어링](#chapter6-3)
    * [Chapter 6-3-1 프롬프트의 잠재적인 복잡성](#chapter6-3-1)       
    * [Chapter 6-3-2 문맥 내 학습: 예시제공](#chapter6-3-2)     
    * [Chapter 6-3-3 프롬프트 체인: 문제 쪼개기](#chapter6-3-3)  
* [Chapter 6-4 생성 모델을 사용한 추론](#chapter6-4)    
    * [Chapter 6-4-1 CoT: 답변하기 전에 생각하기](#chapter6-4-1)    
    * [Chapter 6-4-2 자기 일관성: 출력 샘플링](#chapter6-4-2)  
    * [Chapter 6-4-3 ToT: 중간 단계 탐색](#chapter6-4-3)  
* [Chapter 6-5 출력 검증](#chapter6-5)        
    * [Chapter 6-5-1 예시 제공](#chapter6-5-1)        

## Chapter 6 서론 <a class="anchor" id="chapter6"></a>
1. 사전 훈련된 트랜스포머 기반 모델은 사용자의 프롬프트에 대한 응답으로 텍스트를 생성하는 능력이 뛰어나다.

2. 프롬프트 엔지니어링은 생성된 텍스트의 품질을 향상시키기 위해 프롬프트를 설계하는 방법이다.

## Chapter 6-1 텍스트 생성 모델 사용하기 <a class="anchor" id="chapter6-1"></a>
### Chapter 6-1-1 텍스트 생성 모델 선택하기 <a class="anchor" id="chapter6-1-1"></a>
1. 텍스트 모델을 선택하기 전에 독점 모델을 사용할지 오픈 소스 모델을 사용할 지 결정해야한다.
    - 독점 모델은 일반적으로 더 강력하지만 비용이 많이 들고, 오픈 소스 모델은 더 유연하고 자유롭게 사용할 수 있다.

2. 일반적으로 유명한 파운데이션 모델은 아래의 이미지와 같다.

    ![파운데이션 모델](./image/06_foundation_models.png)

3. 이런 파운데이션 모델을 사용해 미세 튜닝된 모델은 수천개 이상 존재하며, 각각 특정 작업에 최적화 되어있다.
    - 이런 모델 중 어떤 모델을 선택해야하는 지는 사용자의 작업과 요구사항에 따라 다르다.

4. 작은 크기의 파운데이션 모델에서 시작하는 것이 좋다.
    - Phi-3-mini는 작은 VRM이 설치된 장치에서 사용할 수 있다.

5. 일반적으로 작은 모델에서 스케일을 확대하는 것이 축소하는 것보다 용이하다.
    - 작은 모델에서 원하는 결과를 얻을 수 있다면, 더 큰 모델에서도 원하는 결과를 얻을 가능성이 높다.

### Chapter 6-1-2 텍스트 생성 모델 로드하기 <a class="anchor" id="chapter6-1-2"></a>

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Phi-3-mini 모델과 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct"
                                            , device_map="cuda",
                                            dtype="auto")

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# 텍스트 생성 파이프라인 생성
pip = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False, max_new_tokens=500, do_sample=False)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [3]:
# 프롬프트
message = [{"role": "user", "content": "Create a funny joke about chickens"},]

# 텍스트 생성
output = pip(message)
print(output[0]['generated_text'])

 Why did the chicken join the band? Because it had the drumsticks!


In [7]:
# 프롬프트 템플릿을 적용합니다.
prompt = pip.tokenizer.apply_chat_template(message, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens<|end|>
<|endoftext|>


1. 프롬프트 템플릿은 모델을 훈련하는데 사용된다.
    - 누가 무엇을 말했는지에 대한 정보와 모델이 어떻게 응답해야하는지에 대한 정보를 포함한다.

2. 프롬프트 템플릿을 이미지로 표현하면 아래와 같다.

    ![프롬프트 템플릿](./image/06_prompt_template.png)

### Chapter 6-1-3 모델 출력 제어하기 <a class="anchor" id="chapter6-1-3"></a>
1. 모델의 매개변수를 조정하여 출력을 제아할 수 있다.
    - 예를 들어, `temperature` 매개변수는 모델의 출력을 더 창의적이거나 보수적으로 만들 수 있다.
    - `top_k`와 `top_p` 매개변수는 모델이 다음 단어를 선택할 때 고려하는 후보 단어의 수를 제한한다.

2. "I am driving a"란 문장된에 "car"나 "trcuk"과 같은 단어가 나올 확율이 "elephant"보다 높지만, 매우 낮더라도 "elephant"가 생성될 가능성은 있다.

3. do_sample=False로 설정하면 모델이 항상 가장 높은 확률을 가진 단어를 선택하도록 강제할 수 있다.

4. temperature 매개변수는 텍스트 생성의 무작위성 또는 창의성을 조절한다.
    - 확률이 낮은 토큰을 선택할 가능성이 얼마인지 결정한다.
    - 0에 가까운 값은 모델이 가장 확률이 높은 토큰을 선택하도록 강제한다.


In [ ]:
# 높은 temperature로 텍스트 생성
# 실행할 때마다 출력이 바뀐다.
output = pip(message, temperature=1.0, do_sample=True)
print(output[0]['generated_text'])

 Why did the chicken join the symphony orchestra? Because she had good chicken beats!


5. top_p는 LLM이 고려할 토큰 일부를 제어하는 샘플링 기법이다.
    - top_p에 지정한 누적 확률에 도달할 때까지 후보 토큰을 모은다.
    - 0.1로 설정하면 모델이 다음 단어를 선택할 때 가장 확률이 높은 10%의 토큰만 고려한다.
    - 아래의 이미지에서 보듯이 top_p가 낮을수록 모델이 다음 단어를 선택할 때 고려하는 후보 토큰의 수가 줄어든다.
    
        ![top_p](./image/06_top_p.png)

6. top_k는 모델이 고려할 수 있는 토큰의 수를 제어한다.
    - top_k=50으로 설정하면 모델이 다음 단어를 선택할 때 가장 확률이 높은 50개의 토큰만 고려한다.
    - top_k가 낮을수록 모델이 다음 단어를 선택할 때 고려하는 후보 토큰의 수가 줄어든다.

In [9]:
# 높은 top_p로 텍스트 생성
output = pip(message, top_p=1, do_sample=True)
print(output[0]['generated_text'])

 Why did the chicken join a book club? Because it wanted to improve its egg-scape!


7. temperature, top_p 값 사용 예시는 아래의 이미지와 같다.

    ![temperature_top_p](./image/06_temperature_top_p.png)

## Chapter 6-2 프롬프트 엔지니어링 소개 <a class="anchor" id="chapter6-2"></a>
1. 프롬프트 엔지니어링의 주요 목적은 모델로부터 유용한 응답을 얻는 것이다.

2. 프롬프트 최적화는 반복적인 과정이 필요하다.

### Chapter 6-2-1 프롬프트의 기본 구성요소<a class="anchor" id="chapter6-2-1"></a>
1. LLM은 아래의 그림과 같이 프롬프트를 통해 특정 작업을 요청하지 않으면 기본적으로 이전 단어를 기반으로 이후에 나올 단어를 예측한다.

    ![프롬프트 예측](./image/06_prompt_prediction.png)

 2. LLM에 한 문장이 긍정적인지 부정적인지를 분류한다고 가정해보자
    - 프롬프트는 지시와 이에 관련된 두 가지 데이터로 구성될 수 있다.
    
      ![프롬프트 구성요소](./image/06_prompt_components.png)

3. 모델이 '긍정적', '부정적' 만을 출력하게 하려면 프롬프트에 명확한 지시어가 필요하다.
    - 모델은 이런 요소에 직접 훈련되지 않았지만 이런 구조를 일반화할 수 있는 충분한 지시문을 학습했다.
    
        ![프롬프트 지시어](./image/06_prompt_instruction.png)


4. 원하는 응답을 얻을 때까지 프롬프트에 요소를 추가하거나 업데이트할 수 있다.
    - 예시를 추가, 사용 사례 설명, 추가적인 맥락제공 등.
    - 이런 구성 요소를 설계하는 창의설이 핵심이다.

### Chapter 6-2-2 지시 기반 프롬프트<a class="anchor" id="chapter6-2-2"></a>
1. 지시 기반 프롬프트는 모델이 특정 작업을 수행하도록 명확한 지시어를 포함하는 프롬프트이다.
    - 특정 작업에 따라 프롬프트의 양식은 달라질 수 있다.

        ![지시 기반 프롬프트](./image/06_instruction_based_prompt.png)


2. 작업마다 다른 지시가 필요하지만 출력 품질을 샹상시키기 위한 일반적은 지침은 다음과 같다.
    - 구체성: 모델이 무엇을 해야하는지 명확하게 설명한다.
    - 환각: 모델이 답변을 알 때만 생성하라고 지시한다.
    - 순서: 프롬프트의 시작이나 끝에 지시 사항을 전달한다.

## Chapter 6-3 고급 프롬프트 엔지니어링<a class="anchor" id="chapter6-3"></a>
### Chapter 6-3-1 프롬프트의 잠재적인 복잡성<a class="anchor" id="chapter6-3-1"></a>
1. 프롬프트의 고급 구성 요소는 프롬프트를 쉽게 복잡하게 만들 수 있다.
    - 페르소나(정체성): LLM이 수행할 역활을 기술한다.
        - 예시) "당신은 친절한 조수입니다. 당신의 임무는 사용자의 질문에 친절하게 답하는 것입니다."
    - 지시(핵심 작업): LLM이 수행할 작업을 가능한 구제척으로 나타낸다.
        - 달리 해석될 여지가 없는 명확한 지시가 필요하다.
    - 문맥(추가 정보): 문제나 작업의 맥락을 설명하는 추가정보.
        - 예시) "당신의 임무는 사용자의 질문에 친절하게 답하는 것입니다. 사용자는 오늘 날씨가 어떨지 물어봤습니다."
    - 형식(추가 정보): LLM이 텍스트를 출력하는데 사용할 형식.
        - 예시) "사용자는 오늘 날씨가 어떨지 물어봤습니다. 당신의 답변은 JSON 형식이어야 합니다."
    - 청중(누구를 위한것인가): LLM이 텍스트를 생성할 때 고려해야할 청중으로, 출력의 수준과 스타일에 영향을 줄 수 있다.
        - 예시) "당신의 청중은 어린이입니다."
    - 어투(텍스트 스타일): LLM이 텍스트를 생성할 때 사용할 어투로, 출력의 스타일과 톤에 영향을 줄 수 있다.
        - 예시) "당신의 어투는 친절하고 격식적입니다."
    - 데이터(요약할 내용): LLM이 텍스트를 생성할 때 사용할 데이터로, 모델이 텍스트를 생성하는데 필요한 정보를 제공한다.
        - 예시) "당신의 데이터는 오늘 날씨에 대한 정보입니다."
    - 위의 프롬프트를 정리하면 아래와 같다.
         - 예시) "당신은 친절한 조수입니다. 당신의 임무는 사용자의 질문에 친절하게 답하는 것입니다. 사용자는 오늘 날씨가 어떨지 물어봤습니다. 당신의 답변은 JSON 형식이어야 합니다. 당신의 청중은 어린이입니다. 당신의 어투는 친절하고 격식적입니다. 당신의 데이터는 오늘 날씨에 대한 정보입니다."

2. 프롬프트의 복잡성은 모듈식 특징을 보여주어 구성요소를 자유롭게 추가하거나 제거하면서 프롬프트를 조절할 수 있다.

3. 초두 효과와 신근 효과에서 보았듯이 구성 요소의 순서가 출력의 품일에 영향을 줄 수 있다.

### Chapter 6-3-2 문맥 내 학습: 예시제공<a class="anchor" id="chapter6-3-2"></a>
1. LLM에게 우리가 달성하려는 것의 예시를 정확하게 제공하는 방법을 문맥 내 학습이라고 한다.
    - 모델에 얼마나 많은 예시를 보여주는 지에 따라 원샷, 퓨샷등으로 나뉜다.

        ![문맥 내 학습](./image/06_in_context_learning.png)

2. 질문(user)와 답변(assistant)를 구분하지 않는 경우 모델이 질문과 답변을 혼동할 수 있다.
    - 예시 < s ><|user|>What is the capital of France?<|end|><|assistant|>Paris.<|user|>What is the capital of Germany?<|end|><|assistant|>

### Chapter 6-3-3 프롬프트 체인: 문제 쪼개기<a class="anchor" id="chapter6-3-3"></a>
1. 문제를 프롬프트 안에서 분할하는 대신 여러 프롬프트로 문제를 분해할 수 있다.
    - 한 프롬프트의 출력을 다음 프롬프트의 입력으로 사용하는 방법이다.

3. 여러 제품의 특징을 바탕으로 제품 이름, 슬로건, 홍보 문안을 만들고 싶다고 가정한다.
    - 먼저 제품 특징을 사용해 이름을 만들고, 이를 활용해 슬로건을 만든다음, 마지막으로 특징, 이름, 슬로건을 사용해 홍보 문안을 만든다.

        - ![프롬프트 체인](./image/06_prompt_chain.png)

3. 모델을 여러번 호출해야 하지만 각각의 호출마다 매개변수를 다르게 설정할 수 있다.
    - 예시) 제품 이름을 만들 때는 창의적인 출력을 위해 높은 temperature를 사용할 수 있지만, 슬로건을 만들 때는 더 보수적인 출력을 위해 낮은 temperature를 사용할 수 있다.

## Chapter 6-4 생성 모델을 사용한 추론<a class="anchor" id="chapter6-4"></a>
1. 프롬프트 엔지니어링을 통해 LLM과 협업하여 추론 과정을 모방하고 LLM의 출력을 개선할 수 있다.

### Chapter 6-4-1 CoT: 답변하기 전에 생각하기<a class="anchor" id="chapter6-4-1"></a>
1. CoT(Chain of Thought) 프롬프트는 모델이 답변하기 전에 문제에 대한 생각 과정을 생성하도록 지시하는 프롬프트이다.
    - 모델이 응답을 생성하기 전에 수행할 추론을 보여주는 사고 과정 예시를 프롬프트에 추가한다.
    - 수학 문제와 같이 복잡도가 매우 높은 작업에 매우 이롭다.
    - 몇 개의 토큰을 기반으로 전체 해달을 계산하는 대신에 추론 과정에 추가된 토큰을 기반으로 출력을 만든다.
  
        ![CoT](./image/06_cot.png)

2. CoT 논문에 포함된 예시를 보면 모델이 정답뿐만 아니라 답을 내기 전에 설명을 생성하는 것을 볼 수 있다.
    - 예시) "Q: 123 + 456 = ? A: 123 + 456 = 579. 따라서 정답은 579입니다."

3. CoT에 추가하는 추론 예시를 생성 모델에게 수행하도록 지시할 수 있다.
    - 일반적으로 'Let's think step by step'과 같은 프롬프트를 사용한다.

        ![CoT 프롬프트](./image/06_cot_prompt.png)

### Chapter 6-4-2 자기 일관성: 출력 샘플링<a class="anchor" id="chapter6-4-2"></a>
1. temperature와 top_p 같은 매개변수로 일정 수준의 창의성을 허용하면 동일한 프롬프트를 여러번 실행 할때마다 다양한 결과를 얻을 수 있다.
    - 출력의 품질은 향상되거나 저하될 수 있다.

2. 무작위성에 대응하고 생성 품질을 향상시키기 위해 자기 일관성 방법이 개발되었다.
    - 모델이 동일한 프롬프트에 대해 여러 출력을 생성하도록 허용한다.
    - 그런 다음 가장 일관된 답변을 선택한다.
    - 예시) 모델이 동일한 수학 문제에 대해 여러 답변을 생성하도록 허용한 다음, 가장 자주 생성된 답변을 선택한다.

3. 하나의 질문에 여러 번 요청을 해야하므로 성능은 향상되지만 비용이 증가한다.

4. 여러 개의 '사고'에서 샘플링하여 모델을 더 사려 깊게 만듦으로써 생성 모델의 출력을 향상시키는 것이 목표이다.

### Chapter 6-4-3 ToT: 중간 단계 탐색<a class="anchor" id="chapter6-4-3"></a>
1. 여런 단계의 추론이 필요한 문제를 만났을 때 모델은 트리 기반 구조를 활용해 중간 사고를 생성하고 이를 평가한다.
    - 전망이 가장 밝은 사라고를 유지하고 가정 어두운 사고는 삭제한다.
    
2. 이야기를 작성하거나 창의적인 아이디어를 도출하는 것처럼 여러 가지 경로를 탐색하는 것이 유용한 작업에 특히 유용하다.

3. 생성 모델을 여러번 호출하는 대신 모델이 이런 동작을 모방하도록 하여 전문가 여럿이 주고 받는 대화를 흉내내도록 할 수 있다.

4. 프롬프트와 출력은 아래의 이미지와 같다.

    ![ToT](./image/06_tot.png)

## Chapter 6-5 출력 검증<a class="anchor" id="chapter6-5"></a>
1. 출력을 검증하는 이유는 다음과 같다.
    - 구조적인 출력: 모델이 특정 형식으로 출력을 생성하도록 요구하는 경우, 출력이 올바른 형식을 따르는지 검증해야한다.
    - 유요한 출력: 모델이 특정 작업에 대한 유효한 출력을 생성하도록 요구하는 경우, 출력이 유효한지 검증해야한다.
    - 윤리: 욕설, 개인 식별 정보, 편향등과 같은 부적절한 출력을 방지하기 위해 출력이 윤리적인지 검증해야한다.
    - 정확성: 모델이 사실에 기반한 출력을 생성하도록 요구하는 경우, 출력이 정확한지 검증해야한다.

2. 일반적으로 생성 모델의 출력을 제어하는 방법은 세 가지이다.
    - 예시: 기대하는 출력의 예시를 여러개 제공한다.
    - 문법: 토큰 선택 제한과 같은 문법적 제약을 사용하여 모델이 특정 형식으로 출력을 생성하도록 강제한다.
    - 미세 튜닝: 기대 출력이 포함된 데이터에서 모델을 튜닝한다.

### Chapter 6-5-1 예시 제공<a class="anchor" id="chapter6-5-1"></a>
